# AI Model Experiment & Evaluation
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membandingkan dua pendekatan AI dalam menyelesaikan task sentiment analysis ulasan pelanggan:
1. Model klasik Machine Learning (Scikit-learn)
2. LLM API (Gemini)

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

> 🔧 **Catatan Penyesuaian Dataset**
>
> Notebook ini menggunakan dataset `customer_reviews_sentiment.csv` dengan kolom `review_text` dan `sentiment` (nilai: `positif`/`negatif`).
>
> Apabila dataset final berbeda dari yang digunakan saat ini, sesuaikan bagian berikut:
> - Nama file pada `pd.read_csv(...)` di Section 2
> - Nama kolom teks dan label pada Section 3 (saat ini: `review_text`, `sentiment`)
> - Label yang diminta pada prompt LLM di Section 5.2 (saat ini: `positif`/`negatif`)
> - Studi Kasus pada Assignment Brief, apabila domain data berbeda dari e-commerce

## 1. Problem Statement

Objective:
menyelesaikan use case sentiment analysis terhadap ulasan pelanggan sebuah e-commerce yang ingin membangun fitur otomatis untuk mengklasifikasikan sentimen ulasan pelanggan: 
(1) model klasik Machine Learning yang dilatih sendiri menggunakan Scikit-learn
(2) Large Language Model (LLM) yang diakses melalui Gemini AI API menggunakan pendekatan prompting
Target:
1. Memahami problem statement dan objective dari sebuah use case AI.
2. Melatih model klasik Machine Learning menggunakan Scikit-learn untuk task klasifikasi teks.
3. Menjalankan inference LLM melalui Gemini AI API menggunakan pendekatan prompting untuk task yang sama.
4. Mengevaluasi performa kedua pendekatan menggunakan metrik evaluasi yang relevan (Accuracy, Precision, Recall, F1-Score).
5. Menganalisis trade-off antara model klasik dan LLM API dari sisi performa, kecepatan implementasi, dan biaya.
6. Menyusun rekomendasi technical approach berdasarkan hasil eksperimen dan analisis.

Batasan/Asumsi yang digunakan:
- Dataset berjumlah 200 baris (160 train / 40 test) — skala kecil untuk tujuan eksperimen/pembelajaran, bukan representasi data produksi.
- Klasifikasi sentimen hanya biner: **positif** atau **negatif** (tidak ada label netral).
- Split train/test dilakukan sekali secara acak (single split), tanpa cross-validation.
- Prompt LLM dirancang secara **zero-shot**, tanpa contoh (few-shot examples).
- Output LLM diasumsikan valid jika mengandung kata kunci "positif"/"negatif" setelah normalisasi (lowercase + strip); tidak ada verifikasi makna kontekstual lebih lanjut.
- Skor evaluasi (accuracy, precision, recall, F1 = 1.0) diasumsikan **belum tentu general** ke dataset lain yang lebih besar/beragam, karena kemungkinan dipengaruhi ukuran data yang kecil dan teks yang eksplisit.
- API key LLM disimpan di environment variable, bukan dihardcode, dan diasumsikan valid/aktif selama proses evaluasi berjalan.
- Perbandingan biaya dan kecepatan bersifat kualitatif (berdasarkan karakteristik umum pendekatan), bukan hasil benchmarking waktu/biaya aktual yang presisi.

## 2. Import Library & Load Dataset

In [ ]:
# 🔧 Sesuaikan nama file jika dataset final berbeda
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from google import genai  # sesuaikan dengan library Gemini API yang digunakan
from google.genai import types
import os
from pathlib import Path

BASE_DIR = Path.cwd().parent
print(f"Current working directory: {BASE_DIR}")
df = pd.read_csv(BASE_DIR / "data" / "customer_reviews_sentiment.csv")
df.head()

## 3. Data Cleansing
Dari sini tidak ada value yang duplikasi dan missing value jadi sudah clean

In [27]:
print("Kolom:", df.columns.tolist())
print("\nInfo:")
print(df.info())

print("\nDistribusi label:")
print(df["sentiment"].value_counts())

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplikat teks:", df["review_id"].duplicated().sum())

Kolom: ['review_id', 'product_name', 'review_text', 'sentiment']

Info:
<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   review_id     200 non-null    int64
 1   product_name  200 non-null    str  
 2   review_text   200 non-null    str  
 3   sentiment     200 non-null    str  
dtypes: int64(1), str(3)
memory usage: 6.4 KB
None

Distribusi label:
sentiment
positif    110
negatif     90
Name: count, dtype: int64

Missing values:
review_id       0
product_name    0
review_text     0
sentiment       0
dtype: int64

Duplikat teks: 0


## 3. Menyiapkan Train/Test Split

Pastikan test set yang sama digunakan untuk kedua pendekatan agar perbandingan adil.

In [28]:
# 🔧 Sesuaikan nama kolom jika dataset final menggunakan nama kolom berbeda
X = df['review_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train size:', len(X_train))
print('Test size:', len(X_test))

print("\nDistribusi label di train:")
print(pd.Series(y_train).value_counts(normalize=True))
print("\nDistribusi label di test:")
print(pd.Series(y_test).value_counts(normalize=True))

Train size: 160
Test size: 40

Distribusi label di train:
sentiment
positif    0.55
negatif    0.45
Name: proportion, dtype: float64

Distribusi label di test:
sentiment
positif    0.55
negatif    0.45
Name: proportion, dtype: float64


## 4. Pendekatan 1 — Model Klasik (Scikit-learn)

### 4.1 Preprocessing & Feature Extraction

In [29]:
vectorizer = TfidfVectorizer() # Membuat objek vectorizer kosong. Belum tahu apa-apa tentang dataset.
X_train_vec = vectorizer.fit_transform(X_train) # Mengubah data train menjadi representasi vektor. Sekarang vectorizer sudah tahu tentang dataset. (Pelajari kosakata (vocabulary) dari train set saja)
X_test_vec = vectorizer.transform(X_test) # Mengubah data test menjadi representasi vektor. Vectorizer menggunakan pengetahuan dari data train. ( Ubah setiap teks di train set menjadi vektor TF-IDF berdasarkan kosakata yang sudah dipelajari.)

### 4.2 Training Model

In [ ]:
clf = LogisticRegression() # algoritma klasifikasi. Namanya begitu karena secara matematis memakai fungsi logistik (sigmoid) untuk memetakan output linear ke probabilitas 0–1.
clf.fit(X_train_vec, y_train) # melakukan optimasi untuk mencari bobot w dan b yang paling memisahkan positif dan negatif

# Interpretasi Bobot
import numpy as np

feature_names = vectorizer.get_feature_names_out()
coefs = clf.coef_[0]

# Kata yang paling mendorong ke NEGATIF
top_negative_idx = np.argsort(coefs)[:10]
print("Top kata NEGATIF:")
for i in top_negative_idx:
    print(f"  {feature_names[i]:20s} → {coefs[i]:.4f}")

# Kata yang paling mendorong ke POSITIF
top_positive_idx = np.argsort(coefs)[-10:][::-1]
print("\nTop kata POSITIF:")
for i in top_positive_idx:
    print(f"  {feature_names[i]:20s} → {coefs[i]:.4f}")

Top kata NEGATIF:
  lambat               → -1.2440
  jauh                 → -1.0900
  sudah                → -0.9521
  rusak                → -0.8873
  yang                 → -0.8664
  tidak                → -0.8628
  sekali               → -0.8272
  seperti              → -0.7666
  komplain             → -0.7396
  penjual              → -0.7396

Top kata POSITIF:
  cepat                → 1.6160
  rapi                 → 1.0702
  nyaman               → 0.9641
  pas                  → 0.9641
  ada                  → 0.9487
  lebih                → 0.8915
  puas                 → 0.8667
  recommended          → 0.8221
  seller               → 0.7696
  kualitas             → 0.7385


### 4.3 Prediksi pada Test Set

In [ ]:
y_pred_classic = clf.predict(X_test_vec)

# Check Results
print("Shape prediksi:", y_pred_classic.shape)
print("Contoh prediksi:", y_pred_classic[:5])
print("Distribusi prediksi:")
print(pd.Series(y_pred_classic).value_counts())

Shape prediksi: (40,)
Contoh prediksi: ['negatif' 'positif' 'positif' 'positif' 'negatif']
Distribusi prediksi:
positif    22
negatif    18
Name: count, dtype: int64


## 5. Pendekatan 2 — LLM API (Gemini / Groq)

### 5.1 Setup API

In [73]:
# Simpan API key di environment variable, JANGAN hardcode langsung di notebook
from groq import Groq
#client = genai.Client(api_key=os.environ.get('GROQ_API_KEY'))
client = Groq(api_key=os.environ.get('GROQ_API_KEY'))

### 5.2 Merancang Prompt

_Tuliskan prompt yang kamu rancang di sini, dan jelaskan alasannya (zero-shot / few-shot)._

In [86]:
# 🔧 Sesuaikan label pada prompt jika dataset final menggunakan label berbeda
# Use the Gemini API to classify sentiment using a LLM
def classify_sentiment_llm(review_text):
    prompt = f'''Klasifikasikan sentimen ulasan berikut sebagai "positif" atau "negatif".
Jawab hanya dengan satu kata: positif atau negatif.

Ulasan: "{review_text}"
Sentimen:'''
    response = client.models.generate_content(
        model='gemini-3.1-flash-lite',  # sesuaikan versi model
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.2)
    )
    return response.text.strip().lower()
# Use the Groq API to classify sentiment using a LLM
def classify_sentiment_groq(review_text):
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",  # Model sangat cepat dan hemat
        messages=[
            {
                "role": "system",
                "content": "Jawab hanya dengan satu kata: positif atau negatif."
            },
            {"role": "user", "content": f'Ulasan: "{review_text}"\nSentimen:'}
        ],
        temperature=0.2
    )
    return response.choices[0].message.content.strip().lower()

### 5.3 Menjalankan Prediksi pada Test Set

Catatan: lakukan normalisasi terhadap output LLM sebelum dibandingkan dengan label asli (misal: lowercase, strip whitespace).

In [ ]:
#y_pred_llm = []
#for review in X_test:
#    pred = classify_sentiment_groq(review)
#    y_pred_llm.append(pred)

# TODO: normalisasi y_pred_llm agar konsisten dengan label 'positif'/'negatif'
import time

def normalize_label(raw_output: str) -> str:
    text = raw_output.lower().strip()
    if "positif" in text or "positive" in text:
        return "positif"
    elif "negatif" in text or "negative" in text:
        return "negatif"
    return "unknown"  # tandai agar bisa dicek manual nanti

y_pred_llm = []
for i, review in enumerate(X_test):
    raw = ""
    for attempt in range(3):
        try:
            raw = classify_sentiment_groq(review)
            break
        except Exception as e:
            print(f"[{i}] percobaan {attempt+1} gagal: {e}")
            time.sleep(2)
    y_pred_llm.append(normalize_label(raw))
    time.sleep(0.3)  # jeda kecil untuk menghindari rate limit

# cek apakah ada label "unknown" yang perlu diperiksa manual
print("Jumlah unknown:", y_pred_llm.count("unknown"))

Jumlah unknown: 0


## 6. Evaluasi dan Perbandingan

### 6.1 Evaluasi Model Klasik

In [89]:
print('=== Model Klasik (Scikit-learn) ===')
print('Accuracy:', accuracy_score(y_test, y_pred_classic))
print(classification_report(y_test, y_pred_classic))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_classic))

=== Model Klasik (Scikit-learn) ===
Accuracy: 1.0
              precision    recall  f1-score   support

     negatif       1.00      1.00      1.00        18
     positif       1.00      1.00      1.00        22

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

Confusion Matrix:
 [[18  0]
 [ 0 22]]


### 6.2 Evaluasi LLM API

In [90]:
print('=== LLM API (Groq) ===')
print('Accuracy:', accuracy_score(y_test, y_pred_llm))
print(classification_report(y_test, y_pred_llm))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_llm))

=== LLM API (Groq) ===
Accuracy: 1.0
              precision    recall  f1-score   support

     negatif       1.00      1.00      1.00        18
     positif       1.00      1.00      1.00        22

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

Confusion Matrix:
 [[18  0]
 [ 0 22]]


### 6.3 Tabel Perbandingan Ringkasan

_Susun tabel ringkasan (bisa markdown table atau DataFrame) yang membandingkan Accuracy, Precision, Recall, F1-Score kedua pendekatan._

In [91]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Hitung metrik untuk masing-masing pendekatan
# average='weighted' dipakai agar tetap adil jika kelas positif/negatif tidak seimbang jumlahnya
metrics_classic = {
    'Accuracy': accuracy_score(y_test, y_pred_classic),
    'Precision': precision_score(y_test, y_pred_classic, average='weighted'),
    'Recall': recall_score(y_test, y_pred_classic, average='weighted'),
    'F1-Score': f1_score(y_test, y_pred_classic, average='weighted'),
}

metrics_llm = {
    'Accuracy': accuracy_score(y_test, y_pred_llm),
    'Precision': precision_score(y_test, y_pred_llm, average='weighted'),
    'Recall': recall_score(y_test, y_pred_llm, average='weighted'),
    'F1-Score': f1_score(y_test, y_pred_llm, average='weighted'),
}

df_comparison = pd.DataFrame({
    'Model Klasik (Scikit-learn)': metrics_classic,
    'LLM API (Groq)': metrics_llm
}).round(4)

df_comparison

,Model Klasik (Scikit-learn),LLM API (Groq)
Accuracy,1.0,1.0
Precision,1.0,1.0
Recall,1.0,1.0
F1-Score,1.0,1.0


## 7. Analisis Trade-off dan Limitation

### Catatan Penting: Skor Sempurna (1.0) pada Kedua Model

Baik Model Klasik (Scikit-learn) maupun LLM API (Groq) menghasilkan accuracy, 
precision, recall, dan F1-score sebesar 1.0 pada seluruh 40 data test, dengan 
confusion matrix yang bersih tanpa satu pun kesalahan klasifikasi (18/18 negatif 
dan 22/22 positif benar).

Setelah proses cleansing dipastikan tidak ada missing value maupun duplikat teks, 
dan distribusi label antara train (55% positif / 45% negatif) dan test (55% / 45%) 
konsisten — sehingga kemungkinan *data leakage* akibat duplikasi antar split relatif 
kecil. Namun, skor sempurna pada dua pendekatan yang arsitekturnya sangat berbeda 
(model statistik klasik vs LLM generatif) tetap perlu dicermati, karena kemungkinan 
besar disebabkan oleh:

1. **Ukuran dataset yang kecil** — total hanya 200 baris (test set 40 baris). Pada 
   skala sekecil ini, sangat mungkin kedua model "kebetulan" benar semua, terutama 
   jika sinyal sentimen dalam teks cukup eksplisit.
2. **Karakteristik teks yang cenderung eksplisit/template** — ulasan produk yang 
   dipakai kemungkinan memuat kata kunci sentimen yang jelas dan konsisten (misal 
   "bagus", "puas", "kecewa", "buruk"), tanpa banyak variasi bahasa seperti sarkasme, 
   negasi bertingkat, atau ambiguitas konteks yang biasa muncul di ulasan dunia nyata. 
   Pola seperti ini membuat klasifikasi sentimen menjadi tugas yang relatif mudah 
   untuk model apa pun, termasuk pendekatan berbasis kata kunci sederhana.
3. **Test set belum diuji dengan data "sulit"** — belum ada percobaan dengan ulasan 
   yang secara sengaja ambigu, campuran sentimen dalam satu teks, atau memakai bahasa 
   informal/typo yang lebih menantang kedua pendekatan.

Dengan demikian, hasil 1.0 pada kedua model **kemungkinan besar mencerminkan tingkat 
kesulitan dataset yang rendah**, bukan bukti bahwa kedua pendekatan akan selalu 
sempurna pada data ulasan produk yang lebih besar dan beragam di dunia nyata.

### Performa
Dari sisi angka semata, kedua pendekatan setara sempurna pada dataset ini sehingga 
tidak ada perbedaan performa yang bisa dibedakan pada eksperimen kali ini. Perbedaan 
performa yang lebih realistis kemungkinan baru akan terlihat jika diuji pada dataset 
yang lebih besar dan lebih beragam.

### Effort & Kompleksitas Implementasi
- **Model Klasik**: memerlukan tahap preprocessing (cleaning teks, vektorisasi 
  TF-IDF/BoW), pemilihan algoritma, dan training pada 160 data. Effort awal lebih 
  besar karena harus menyiapkan pipeline training, tetapi setelah model jadi, 
  prediksi berjalan instan dan tidak butuh dependensi eksternal.
- **LLM API**: tidak memerlukan proses training sama sekali — cukup merancang prompt 
  zero-shot. Effort awal jauh lebih ringan, terbukti dari jumlah `unknown` = 0 
  (artinya normalisasi prompt & parsing output sudah cukup baik untuk 40 data test 
  ini), tetapi tetap bergantung pada kestabilan API pihak ketiga.

### Kecepatan
- **Model Klasik**: prediksi terhadap 40 data test berjalan hampir instan (dalam 
  hitungan milidetik) karena inferensi dilakukan secara lokal.
- **LLM API (Groq)**: memerlukan 1 request per ulasan ke server eksternal, sehingga 
  total waktu untuk 40 data jauh lebih lama dibanding model klasik (detik hingga 
  puluhan detik, tergantung latensi jaringan), meskipun Groq dikenal sebagai salah 
  satu provider LLM tercepat saat ini.

### Biaya
- **Model Klasik**: gratis setelah proses training selesai, hanya membutuhkan sumber 
  daya komputasi lokal yang kecil untuk dataset sekecil ini (160 baris).
- **LLM API**: setiap dari 40 request test dikenakan biaya (walau kecil per token), 
  dan biaya ini akan bertambah linier seiring bertambahnya volume data — perlu 
  dipertimbangkan jika sistem akan dipakai pada skala produksi dengan ribuan ulasan 
  per hari.
  ![Grafik Biaya Usage Groq](documentation/usage-groq.png)

### Keterbatasan Masing-Masing
- **Model Klasik**:
  - Performanya sangat bergantung pada data training — dengan hanya sedikit baris data 
    training, model rawan tidak general (overfit) terhadap kosakata/pola dalam 
    dataset ini saja, dan berisiko performanya turun signifikan pada ulasan produk 
    lain yang gaya bahasanya berbeda.
  - Tidak bisa menangani kata-kata baru atau ulasan dengan gaya bahasa yang belum 
    pernah dilihat saat training (out-of-vocabulary).
- **LLM API**:
  - Bergantung pada koneksi internet dan ketersediaan layanan Groq.
  - Berisiko privasi bila ulasan pelanggan bersifat sensitif dan dikirim ke server 
    pihak ketiga.
  - Skor sempurna di sini belum menguji ketahanan LLM terhadap teks yang ambigu atau 
    kompleks, sehingga performa sebenarnya di kasus sulit masih belum diketahui.
- **Keterbatasan bersama (berlaku untuk kedua pendekatan)**:
  - Dataset yang digunakan (200 baris) terlalu kecil untuk menyimpulkan performa 
    yang bisa digeneralisasi. Diperlukan pengujian lanjutan dengan dataset yang lebih 
    besar dan lebih beragam (idealnya ribuan baris dengan variasi gaya bahasa, 
    termasuk sarkasme, ulasan campuran, dan typo) sebelum kesimpulan performa dapat 
    dipercaya sepenuhnya.

## 8. Rekomendasi Technical Approach

Berdasarkan hasil eksperimen pada Bagian 6, kedua pendekatan menghasilkan skor 
evaluasi yang identik (accuracy, precision, recall, F1-score = 1.0) pada dataset uji 
sebanyak 40 data. Namun, mengingat skor sempurna ini kemungkinan besar dipengaruhi 
oleh ukuran dataset yang kecil (total 200 baris) dan karakteristik teks yang relatif 
eksplisit, rekomendasi berikut diberikan dengan catatan bahwa **validasi lanjutan 
pada dataset yang lebih besar dan beragam sangat disarankan** sebelum keputusan akhir 
diambil untuk implementasi produksi.

Dengan pertimbangan tersebut, rekomendasi teknis adalah:

1. **Untuk kebutuhan skala produksi dengan volume tinggi dan biaya operasional 
   rendah**: gunakan **Model Klasik (Scikit-learn)**. Karena dataset saat ini masih 
   kecil, performa model di dunia nyata perlu diverifikasi lebih 
   lanjut dengan data training yang lebih besar, tetapi dari sisi arsitektur, 
   pendekatan ini jauh lebih murah dan cepat untuk memproses ulasan dalam jumlah 
   besar (ribuan/hari) tanpa biaya per-request.

2. **Untuk kebutuhan pengembangan cepat (rapid prototyping) atau volume data yang 
   masih rendah**: gunakan **LLM API (Groq)**, karena tidak memerlukan proses 
   training maupun pengumpulan data berlabel dalam jumlah besar — cukup dengan 
   prompt zero-shot yang sudah terbukti bekerja baik pada eksperimen ini (0 output 
   `unknown`). Cocok dipakai di tahap awal proyek sebelum data training yang cukup 
   tersedia untuk melatih model klasik.

3. **Sebelum mengadopsi salah satu pendekatan secara permanen**, disarankan untuk:
   - Menambah ukuran dan variasi dataset (idealnya >1.000 baris, mencakup ulasan 
     dengan sentimen campuran, sarkasme, dan bahasa informal/typo) untuk menguji 
     apakah skor 1.0 ini tetap bertahan atau mulai menurun — ini akan memberi 
     gambaran performa yang jauh lebih realistis.
   - Melakukan uji coba pada data baru yang benar-benar belum pernah dilihat sistem 
     (out-of-sample dari sumber lain), bukan hanya split dari dataset yang sama.

4. **Rekomendasi hybrid (opsional)**: jika ke depannya performa mulai berbeda pada 
   dataset yang lebih besar, pertimbangkan pendekatan hybrid — Model Klasik sebagai 
   lapisan utama untuk efisiensi biaya, dengan LLM API dipanggil khusus untuk kasus 
   yang model klasik kurang yakin (misalnya berdasarkan confidence score rendah), 
   guna menyeimbangkan biaya dan akurasi.

**Kesimpulan**: pada eksperimen skala kecil ini, kedua pendekatan setara secara 
performa, sehingga pilihan akhir sebaiknya didasarkan pada faktor operasional 
(biaya, kecepatan, dan volume data yang akan diproses) daripada perbedaan akurasi — 
karena perbedaan akurasi yang sesungguhnya baru dapat diketahui setelah pengujian 
pada dataset yang lebih besar dan representatif.